In [0]:
%pip install --quiet feedparser httpx faster-whisper
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from typing import Optional

import feedparser
import httpx
from faster_whisper import WhisperModel


In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
# =============================================================================
# Configuração
# =============================================================================

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

FEED_URL = "https://feeds.buzzsprout.com/1793509.rss"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}/ENERGIA"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

# Pasta temporária local pra baixar o áudio antes de transcrever
# (faster-whisper lê de um caminho de arquivo, não direto de bytes em memória).
PASTA_AUDIO_TEMP = "/tmp/minutomega_audio"
os.makedirs(PASTA_AUDIO_TEMP, exist_ok=True)

# Modelo Whisper baixado uma vez, salvo aqui — evita depender de rede
# externa (Hugging Face) toda vez que o job rodar.
PASTA_MODELO_WHISPER = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/whisper_models"
os.makedirs(PASTA_MODELO_WHISPER, exist_ok=True)

JANELA_HORAS = 24

SOURCE_ID = "minutomega"
SOURCE_DESCRICAO = "Linked from MinutoMega (MegaWhat) — transcrição automática"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
]

# Modelo do faster-whisper.
WHISPER_MODEL_SIZE = "medium"
WHISPER_DEVICE = "cpu"       # troque pra "cuda" se o cluster tiver GPU
WHISPER_COMPUTE_TYPE = "int8"  # quantização — mais rápido em CPU, leve perda de precisão

MIN_CHARS_TEXTO = 200

[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def parsear_data_rss(data_str: str) -> str:
    try:
        return parsedate_to_datetime(data_str).strftime("%Y-%m-%d")
    except Exception:
        return HOJE


def dentro_da_janela(entry, horas: int = JANELA_HORAS) -> bool:
    if not entry.get("published_parsed"):
        return True
    pub_dt = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
    limite = datetime.now(timezone.utc) - timedelta(hours=horas)
    return pub_dt >= limite


In [0]:
# =============================================================================
# Etapa 1 — Buscar episódios do feed dentro da janela
# =============================================================================

def obter_episodios_recentes(feed_url: str) -> list:
    resp = httpx.get(feed_url, timeout=20, follow_redirects=True,
                      headers={"User-Agent": random.choice(USER_AGENTS)})
    resp.raise_for_status()

    parsed = feedparser.parse(resp.content)
    print(f"[feed] {len(parsed.entries)} episódios no feed bruto.")

    recentes = [e for e in parsed.entries if dentro_da_janela(e)]
    print(f"[filtro] janela de {JANELA_HORAS}h: {len(parsed.entries)} -> {len(recentes)} episódios recentes.")

    return recentes


def extrair_url_audio(entry) -> Optional[str]:
    """A URL do arquivo de áudio vem na tag <enclosure> do item RSS."""
    for link in entry.get("links", []):
        if link.get("rel") == "enclosure" or "audio" in link.get("type", ""):
            return link.get("href")
    return None


In [0]:
# =============================================================================
# Etapa 2 — Baixar o áudio
# =============================================================================

def baixar_audio(url: str, caminho_destino: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        try:
            with httpx.stream("GET", url, timeout=60, follow_redirects=True,
                               headers={"User-Agent": random.choice(USER_AGENTS)}) as resp:
                if resp.status_code != 200:
                    print(f"  [audio tent {tentativa}/{tentativas}] status={resp.status_code}")
                    continue
                with open(caminho_destino, "wb") as f:
                    for chunk in resp.iter_bytes(chunk_size=8192):
                        f.write(chunk)
            return caminho_destino
        except Exception as e:
            print(f"  [audio tent {tentativa}/{tentativas}] erro: {e}")
            if tentativa < tentativas:
                time.sleep(random.uniform(1.0, 2.5))

    return None


In [0]:
# =============================================================================
# Etapa 3 — Transcrever com faster-whisper
# =============================================================================
# Carrega o modelo uma vez só (reaproveitado entre episódios da mesma execução).

_modelo_whisper: Optional[WhisperModel] = None


def obter_modelo_whisper(tentativas: int = 3) -> WhisperModel:
    global _modelo_whisper
    if _modelo_whisper is not None:
        return _modelo_whisper

    for tentativa in range(1, tentativas + 1):
        try:
            print(f"[whisper] carregando modelo '{WHISPER_MODEL_SIZE}' "
                  f"(tentativa {tentativa}/{tentativas})...")
            _modelo_whisper = WhisperModel(
                WHISPER_MODEL_SIZE,
                device=WHISPER_DEVICE,
                compute_type=WHISPER_COMPUTE_TYPE,
                download_root=PASTA_MODELO_WHISPER,
            )
            return _modelo_whisper
        except Exception as e:
            print(f"  -> falhou: {e}")
            if tentativa < tentativas:
                time.sleep(random.uniform(5.0, 10.0))

    raise RuntimeError("Não foi possível carregar o modelo Whisper após múltiplas tentativas.")


def transcrever_audio(caminho_audio: str) -> str:
    modelo = obter_modelo_whisper()
    inicio = time.time()

    segments, info = modelo.transcribe(caminho_audio, language="pt", beam_size=5)
    texto = " ".join(seg.text.strip() for seg in segments)

    duracao_transcricao = time.time() - inicio
    print(f"    [whisper] transcrito em {duracao_transcricao:.1f}s "
          f"(áudio original: {info.duration:.1f}s)")

    return texto.strip()

In [0]:
# =============================================================================
# Etapa 4 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(SOURCE_ID, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


In [0]:
# =============================================================================
# Etapa 5 — Pipeline principal
# =============================================================================

def processar_episodio(entry) -> Optional[dict]:
    titulo = entry.get("title", "sem-titulo")
    url_pagina = entry.get("link")
    print(f"\n  [episódio] {titulo[:100]}")

    url_audio = extrair_url_audio(entry)
    if not url_audio:
        print("    -> sem <enclosure> de áudio; pulando.")
        return None

    caminho_audio = os.path.join(PASTA_AUDIO_TEMP, f"{hash_curto(url_audio)}.mp3")
    if not baixar_audio(url_audio, caminho_audio):
        print("    -> download do áudio falhou; pulando.")
        return None

    texto = transcrever_audio(caminho_audio)

    # Limpeza do arquivo de áudio temporário — não precisa persistir.
    try:
        os.remove(caminho_audio)
    except OSError:
        pass

    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> transcrição muito curta ({len(texto)} chars); pulando.")
        return None

    data_publicacao = parsear_data_rss(entry.get("published", ""))

    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": SOURCE_DESCRICAO,
        "url": url_pagina or url_audio,
        "date": HOJE,
        "published_at": data_publicacao,
    }

    caminho_txt, caminho_json = salvar_artefatos(PASTA_DESTINO, titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")

    return {"titulo": titulo, "url": url_pagina, "caminho_txt": caminho_txt, "caminho_json": caminho_json}


In [0]:
# =============================================================================
# Execução
# =============================================================================

try:
    episodios = obter_episodios_recentes(FEED_URL)
    print(f"\n=== {len(episodios)} episódio(s) a processar ===")

    todos_resultados: list[dict] = []
    for entry in episodios:
        try:
            resultado = processar_episodio(entry)
            if resultado:
                todos_resultados.append(resultado)
        except Exception as e:
            print(f"[ERRO] episódio {entry.get('title', '?')!r} falhou: {e}")

    print(f"\n\n=== Fim. {len(todos_resultados)} episódio(s) transcrito(s) e salvos em {PASTA_DESTINO} ===")

    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=True,
        docs_capturados=len(todos_resultados),
    )

except Exception as e:
    print(f"\n\n=== ERRO GERAL: {e} ===")
    atualizar_status_fonte(
        source_id=SOURCE_ID,
        sucesso=False,
        docs_capturados=0,
        erro=str(e),
    )


[feed] 1209 episódios no feed bruto.
[filtro] janela de 24h: 1209 -> 1 episódios recentes.

=== 1 episódio(s) a processar ===

  [episódio] Por que a GNA vê risco na abertura dos terminais de GNL?
[whisper] carregando modelo 'medium' (tentativa 1/3)...
    [whisper] transcrito em 1028.4s (áudio original: 1824.1s)
    -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28/minutomega_por-que-a-gna-ve-risco-na-abertura-dos-terminais-de-gnl_969e64e1.txt


=== Fim. 1 episódio(s) transcrito(s) e salvos em /Volumes/desafio_kinea/research/research_volume/infraestrutura/files/2026-07-28 ===
